# GC-AI quickstart

A diagram to SBOL3, and the result into a registry. The same work the
scripts in this directory do, with the detections on screen.

Needs a running stack and `uv sync --extra dev --extra db` in `client/`.
The last section also needs an [SBOL DB](https://github.com/marpaia/sbol-db)
registry.

In [ ]:
import os
import sys
import tempfile
import uuid
from pathlib import Path

from IPython.display import Image, display

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "docs" / "examples").is_dir())
sys.path.insert(0, str(REPO_ROOT / "examples"))

import image_to_sbol
import publish_to_registry
from gcai import GCAIClient

CIRCUIT = REPO_ROOT / "docs" / "examples" / "YES GATE" / "YES-GATE-CIRCUIT.png"
NAMESPACE = "http://sbols.org/unspecified_namespace"
OUT = Path(os.environ.get("GCAI_NOTEBOOK_OUT") or tempfile.mkdtemp(prefix="gcai-quickstart-"))

client = GCAIClient()  # GCAI_URL and private credential-file configuration
assert client.health(), "GC-AI dependencies are not ready"

## What the detector sees

The input is a photograph of a pen-on-paper drawing. `run` prints the
glyphs in 5' to 3' order and writes the image with the boxes drawn on it.

In [ ]:
summary = image_to_sbol.run(
    client,
    CIRCUIT,
    name="YesGate",
    namespace=NAMESPACE,
    out=OUT,
    annotated=OUT / "annotated.png",
)

In [ ]:
display(Image(filename=str(OUT / "annotated.png")))

The circuit drawn there has eight parts. Two were found. The boxes show
which, and `docs/examples/README.md` has the count for each committed
example. Detection is a starting point, not the design.

A diagram also says nothing about which registry parts it draws, so each
part is named after its role and carries no sequence. The document records
the role and the position, which is what was drawn.

In [ ]:
summary.parts, summary.without_sequence, [p.name for p in summary.written]

## Naming the parts

One name per part, in the same order, binds them to parts the service's
registry holds. Their sequences come with them, so GenBank and FASTA can
be produced as well. An empty entry falls back to the role.

In [ ]:
part_names = os.environ.get("GCAI_NOTEBOOK_PART_NAMES", "BBa_R0010,BBa_B0034")
named = None
if part_names:
    named = image_to_sbol.run(
        client,
        CIRCUIT,
        name="YesGateNamed",
        namespace=NAMESPACE,
        names=part_names.split(","),
        out=OUT,
    )
    print((OUT / "YesGateNamed.fasta").read_text())
else:
    print("Named registry-part example skipped: no GCAI_NOTEBOOK_PART_NAMES")


In [ ]:
# FASTA is displayed by the named-parts cell above.


## Into a registry

The registry holds SBOL3, validates every import and keeps the graph
whatever the validator says. It serves an object in either SBOL
vocabulary, so the SBOL2 form below is a fetch, not a local conversion.

This writes to a service other people can read, which is why the script's
command line asks for `--yes`.
Publishing requires `GCAI_NOTEBOOK_PUBLISH=1` and `SBOL_DB_URL` for a disposable
review registry. The notebook uses a unique graph and removes it afterward.


In [ ]:
# Publishing is optional and requires an explicit opt-in to a disposable registry.
if os.environ.get("GCAI_NOTEBOOK_PUBLISH") == "1":
    from gcai import storage

    assert named is not None, "Run the named-parts example before publishing"
    registry_url = os.environ["SBOL_DB_URL"]
    graph = f"https://gc-ai.test/quickstart/{uuid.uuid4()}"
    try:
        publish_to_registry.run(
            registry_url,
            OUT / "YesGateNamed.xml",
            iri=graph,
            design="YesGateNamed",
            namespace=NAMESPACE,
        )
    finally:
        with storage.client(registry_url) as registry:
            registry.delete_graph_by_document_iri(graph)
else:
    print("Registry publishing skipped: set GCAI_NOTEBOOK_PUBLISH=1 to opt in")


In [ ]:
client.close()

## From an agent

`gcai-mcp` does the first part of this as a single MCP tool call:

```sh
claude mcp add --transport stdio gc-ai --scope project -- \
  uv run --with "gc-ai-client[mcp]" gcai-mcp --url http://localhost:5001 --allow-root .
```

The MCP section of [`../client/README.md`](../client/README.md) has the rest.